# Session 1: FHIR Fundamentals with LLM-Assisted Code Generation
# BACKUP VERSION — Local FHIR Server with Cached Synthea Data

> **Why this notebook?** The public SMART FHIR server (`launch.smarthealthit.org`)
> is occasionally unavailable. This backup runs a local mini-FHIR server inside
> the notebook with real Synthea patient data pre-loaded. Your code will be
> identical — the only difference is `FHIR_BASE` points to `localhost:5050`.

## Clinical Scenario
**Find patients with Type 2 diabetes, retrieve their most recent HbA1c values, and identify those with poor glycemic control (HbA1c > 7.5%).**

## How This Notebook Works
- **Pre-built cells** (with code): Just run them with Shift+Enter
- **Empty cells** (with instructions above): Ask Claude to generate the code, paste it in, run it
- **Verification cells**: Run after your code to check your results
- **Markdown cells with ✏️**: Read and fill in where prompted

## What You'll Learn
- How FHIR represents clinical data (Resources, References, Bundles)
- How to query a FHIR server using Python
- How to use an LLM to generate working API calls
- Why answering clinical questions requires multi-step FHIR queries

### 📋 Clinical Code Reference

| Code | System | Meaning | Used In | ICD-10 Reference |
|------|--------|---------|---------|------------------|
| 44054006 | SNOMED CT | Type 2 Diabetes Mellitus | Condition search | E11 |
| 59621000 | SNOMED CT | Essential Hypertension | (Session 3) | I10 |
| 4548-4 | LOINC | Hemoglobin A1c (HbA1c) | Observation search | - |
| 85354-9 | LOINC | Blood Pressure panel | (Session 3) | - |
| 2160-0 | LOINC | Creatinine [Mass/volume] in Serum or Plasma | (Session 3) | - |

**HbA1c Interpretation:**
- < 5.7%: Normal
- 5.7% – 6.4%: Prediabetes
- ≥ 6.5%: Diabetes
- \> 7.0%: Poor glycemic control — needs intervention

**Note on threshold:** In clinical practice the threshold for "poor control" varies by guideline and patient context (commonly 7.0%–9.0%). We use > 7.0% here because the Synthea-generated data on this server skews toward lower HbA1c values, and a higher threshold would yield very few flagged patients. This is an artifact of the synthetic data, not a clinical recommendation.

### 🔗 How FHIR Resources Link Together

Our clinical question requires THREE types of FHIR resources:

```
Condition (code: 44054006 — Type 2 Diabetes, SNOMED CT)
  └─ subject.reference ──→ Patient/{id}
                              ↑
Observation (code: 4548-4 — HbA1c, LOINC)
  └─ subject ─────────────────┘
```

- **Condition** records a diagnosis. It points to the Patient via `subject.reference`.
- **Patient** holds demographics (name, birthdate, gender).
- **Observation** records a lab result or vital sign. It also points to the Patient.

To answer "which diabetic patients have poor HbA1c control?" we must:
1. Search Conditions → find patients with diabetes
2. Follow references → get Patient demographics
3. Search Observations → get each patient's HbA1c values
4. Combine and analyze

In [ ]:
# ============================================================
# SETUP — Run this cell first (starts local FHIR server)
# ============================================================
!pip install -q flask

import requests
import json
import threading
import time
import pandas as pd
from IPython.display import display, HTML
from flask import Flask, request as flask_request, jsonify

# ---- Cached FHIR data (30 Synthea patients with Type 2 Diabetes) ----
FHIR_DATA = {
  "patients": {
    "948e608d-e408-4788-8cf2-8157c49c2940": {
      "resourceType": "Patient",
      "id": "948e608d-e408-4788-8cf2-8157c49c2940",
      "name": [
        {
          "use": "official",
          "given": [
            "Dirk"
          ],
          "family": "Kuhlman"
        }
      ],
      "birthDate": "1971-12-07",
      "gender": "male"
    },
    "1ac8947d-038f-4cc7-81fa-a32e694187e8": {
      "resourceType": "Patient",
      "id": "1ac8947d-038f-4cc7-81fa-a32e694187e8",
      "name": [
        {
          "use": "official",
          "given": [
            "Vanda"
          ],
          "family": "Franecki"
        }
      ],
      "birthDate": "1948-04-26",
      "gender": "female"
    },
    "881f534f-d041-425d-a542-cbf669f43e18": {
      "resourceType": "Patient",
      "id": "881f534f-d041-425d-a542-cbf669f43e18",
      "name": [
        {
          "use": "official",
          "given": [
            "Raeann"
          ],
          "family": "O'Kon"
        }
      ],
      "birthDate": "1968-10-09",
      "gender": "female"
    },
    "bd6f9f37-7295-4cd6-b212-134afcef1253": {
      "resourceType": "Patient",
      "id": "bd6f9f37-7295-4cd6-b212-134afcef1253",
      "name": [
        {
          "use": "official",
          "given": [
            "Josue"
          ],
          "family": "Sipes"
        }
      ],
      "birthDate": "1956-03-10",
      "gender": "male"
    },
    "fa064acf-b7f1-4279-83d3-7a94686da7ba": {
      "resourceType": "Patient",
      "id": "fa064acf-b7f1-4279-83d3-7a94686da7ba",
      "name": [
        {
          "use": "official",
          "given": [
            "Jeffery"
          ],
          "family": "Daniel"
        }
      ],
      "birthDate": "1920-04-23",
      "gender": "male"
    },
    "7099b4c5-6f47-4293-9690-f2afb23b9dd6": {
      "resourceType": "Patient",
      "id": "7099b4c5-6f47-4293-9690-f2afb23b9dd6",
      "name": [
        {
          "use": "official",
          "given": [
            "Willis"
          ],
          "family": "Crona"
        }
      ],
      "birthDate": "1946-10-19",
      "gender": "male"
    },
    "af0e5952-c2ac-44fc-a896-36e9e30c2097": {
      "resourceType": "Patient",
      "id": "af0e5952-c2ac-44fc-a896-36e9e30c2097",
      "name": [
        {
          "use": "official",
          "given": [
            "Madalyn"
          ],
          "family": "Frami"
        }
      ],
      "birthDate": "1988-02-18",
      "gender": "female"
    },
    "efe58fa2-c6df-4a71-8e37-06c5e4f1e2b8": {
      "resourceType": "Patient",
      "id": "efe58fa2-c6df-4a71-8e37-06c5e4f1e2b8",
      "name": [
        {
          "use": "official",
          "given": [
            "Emery"
          ],
          "family": "Purdy"
        }
      ],
      "birthDate": "1967-07-27",
      "gender": "male"
    },
    "ffd502c9-23e1-4f8f-bc8a-87373acad280": {
      "resourceType": "Patient",
      "id": "ffd502c9-23e1-4f8f-bc8a-87373acad280",
      "name": [
        {
          "use": "official",
          "given": [
            "Renato"
          ],
          "family": "Dare"
        }
      ],
      "birthDate": "1976-06-14",
      "gender": "male"
    },
    "9eb43ac3-7c1e-4e25-94cd-4b2c43f7234e": {
      "resourceType": "Patient",
      "id": "9eb43ac3-7c1e-4e25-94cd-4b2c43f7234e",
      "name": [
        {
          "use": "official",
          "given": [
            "Estrella"
          ],
          "family": "Daugherty"
        }
      ],
      "birthDate": "1969-08-23",
      "gender": "female"
    },
    "1d4685c4-e83e-4a73-a8e5-147a4c0b4ed4": {
      "resourceType": "Patient",
      "id": "1d4685c4-e83e-4a73-a8e5-147a4c0b4ed4",
      "name": [
        {
          "use": "official",
          "given": [
            "Elijah"
          ],
          "family": "Zieme"
        }
      ],
      "birthDate": "1910-09-19",
      "gender": "male"
    },
    "5db1e775-33a3-47fb-8954-d97cd43e8074": {
      "resourceType": "Patient",
      "id": "5db1e775-33a3-47fb-8954-d97cd43e8074",
      "name": [
        {
          "use": "official",
          "given": [
            "Nikki"
          ],
          "family": "Ferry"
        }
      ],
      "birthDate": "1961-08-27",
      "gender": "female"
    },
    "9bf59715-246f-4ce2-85a6-58c1757f8d21": {
      "resourceType": "Patient",
      "id": "9bf59715-246f-4ce2-85a6-58c1757f8d21",
      "name": [
        {
          "use": "official",
          "given": [
            "Margery"
          ],
          "family": "Gibson"
        }
      ],
      "birthDate": "1948-11-08",
      "gender": "female"
    },
    "f12a1a30-9ce0-48dd-91de-ce5c3fcf8828": {
      "resourceType": "Patient",
      "id": "f12a1a30-9ce0-48dd-91de-ce5c3fcf8828",
      "name": [
        {
          "use": "official",
          "given": [
            "Ernest"
          ],
          "family": "Labadie"
        }
      ],
      "birthDate": "1938-04-09",
      "gender": "male"
    },
    "0e61c3ad-d11e-4080-a6aa-cac89cae4e37": {
      "resourceType": "Patient",
      "id": "0e61c3ad-d11e-4080-a6aa-cac89cae4e37",
      "name": [
        {
          "use": "official",
          "given": [
            "Isaiah"
          ],
          "family": "Schaden"
        }
      ],
      "birthDate": "1953-02-19",
      "gender": "male"
    },
    "d483a480-4a42-428c-bf9b-fdbaddf78e21": {
      "resourceType": "Patient",
      "id": "d483a480-4a42-428c-bf9b-fdbaddf78e21",
      "name": [
        {
          "use": "official",
          "given": [
            "Leonor"
          ],
          "family": "Trujillo"
        }
      ],
      "birthDate": "1960-01-11",
      "gender": "female"
    },
    "12edb5a0-ec71-45b7-aff6-0fef1c382881": {
      "resourceType": "Patient",
      "id": "12edb5a0-ec71-45b7-aff6-0fef1c382881",
      "name": [
        {
          "use": "official",
          "given": [
            "Delbert"
          ],
          "family": "Jaskolski"
        }
      ],
      "birthDate": "1951-09-13",
      "gender": "male"
    },
    "ebb1f0e2-4fa6-4889-a43b-9cda3c737078": {
      "resourceType": "Patient",
      "id": "ebb1f0e2-4fa6-4889-a43b-9cda3c737078",
      "name": [
        {
          "use": "official",
          "given": [
            "Jessi"
          ],
          "family": "Parisian"
        }
      ],
      "birthDate": "1988-02-14",
      "gender": "female"
    },
    "ec6a274e-5090-45c9-b153-e40418f6fa3d": {
      "resourceType": "Patient",
      "id": "ec6a274e-5090-45c9-b153-e40418f6fa3d",
      "name": [
        {
          "use": "official",
          "given": [
            "Parker"
          ],
          "family": "Casper"
        }
      ],
      "birthDate": "1952-07-31",
      "gender": "male"
    },
    "4d58465c-4703-4c36-ab8f-2f935dc4bee7": {
      "resourceType": "Patient",
      "id": "4d58465c-4703-4c36-ab8f-2f935dc4bee7",
      "name": [
        {
          "use": "official",
          "given": [
            "Lakendra"
          ],
          "family": "Roberts"
        }
      ],
      "birthDate": "1962-07-26",
      "gender": "female"
    },
    "5fd069dc-e337-4062-9722-a732c655c17d": {
      "resourceType": "Patient",
      "id": "5fd069dc-e337-4062-9722-a732c655c17d",
      "name": [
        {
          "use": "official",
          "given": [
            "Coleman"
          ],
          "family": "Ortiz"
        }
      ],
      "birthDate": "1915-07-03",
      "gender": "male"
    },
    "bae2bf00-eca1-47fb-bb43-272f8de1449c": {
      "resourceType": "Patient",
      "id": "bae2bf00-eca1-47fb-bb43-272f8de1449c",
      "name": [
        {
          "use": "official",
          "given": [
            "Louis"
          ],
          "family": "Maggio"
        }
      ],
      "birthDate": "1976-01-16",
      "gender": "male"
    },
    "9efca32a-04f9-438a-990c-a287e8b62aac": {
      "resourceType": "Patient",
      "id": "9efca32a-04f9-438a-990c-a287e8b62aac",
      "name": [
        {
          "use": "official",
          "given": [
            "Vito"
          ],
          "family": "Feeney"
        }
      ],
      "birthDate": "1910-09-19",
      "gender": "male"
    },
    "71304c2e-64b0-4b57-91f6-2c1ff771d72c": {
      "resourceType": "Patient",
      "id": "71304c2e-64b0-4b57-91f6-2c1ff771d72c",
      "name": [
        {
          "use": "official",
          "given": [
            "Silvana"
          ],
          "family": "Cremin"
        }
      ],
      "birthDate": "1959-10-01",
      "gender": "female"
    },
    "ecaea95c-46de-4ac9-a58d-4847d1a3e574": {
      "resourceType": "Patient",
      "id": "ecaea95c-46de-4ac9-a58d-4847d1a3e574",
      "name": [
        {
          "use": "official",
          "given": [
            "Stewart"
          ],
          "family": "Stanton"
        }
      ],
      "birthDate": "1915-07-03",
      "gender": "male"
    },
    "75a4631d-2cd8-4c77-afb6-740bad394ea1": {
      "resourceType": "Patient",
      "id": "75a4631d-2cd8-4c77-afb6-740bad394ea1",
      "name": [
        {
          "use": "official",
          "given": [
            "Tristan"
          ],
          "family": "Gleason"
        }
      ],
      "birthDate": "1951-07-30",
      "gender": "male"
    },
    "7f445dad-0907-4a3e-81a5-ad75ecdff752": {
      "resourceType": "Patient",
      "id": "7f445dad-0907-4a3e-81a5-ad75ecdff752",
      "name": [
        {
          "use": "official",
          "given": [
            "Leonel"
          ],
          "family": "Hand"
        }
      ],
      "birthDate": "1963-12-03",
      "gender": "male"
    },
    "132f48ba-1d72-4e0c-a220-91786d7501aa": {
      "resourceType": "Patient",
      "id": "132f48ba-1d72-4e0c-a220-91786d7501aa",
      "name": [
        {
          "use": "official",
          "given": [
            "Neva"
          ],
          "family": "Hyatt"
        }
      ],
      "birthDate": "1970-05-28",
      "gender": "female"
    },
    "3e26303e-a72a-4c60-866f-e5b81a9f1989": {
      "resourceType": "Patient",
      "id": "3e26303e-a72a-4c60-866f-e5b81a9f1989",
      "name": [
        {
          "use": "official",
          "given": [
            "Evelyn"
          ],
          "family": "McCullough"
        }
      ],
      "birthDate": "1922-09-25",
      "gender": "female"
    },
    "148f10e1-2182-4297-ac1f-39bb9eca6f1a": {
      "resourceType": "Patient",
      "id": "148f10e1-2182-4297-ac1f-39bb9eca6f1a",
      "name": [
        {
          "use": "official",
          "given": [
            "Patria"
          ],
          "family": "Schimmel"
        }
      ],
      "birthDate": "1974-05-21",
      "gender": "female"
    }
  },
  "conditions": [
    {
      "resourceType": "Condition",
      "id": "526c0f8c-93db-46e9-b6a3-43dd2409c0c8",
      "subject": {
        "reference": "Patient/ecaea95c-46de-4ac9-a58d-4847d1a3e574"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "1960-09-03T19:20:06+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "681199da-f2e1-45e6-80f2-7a1fb9e668f2",
      "subject": {
        "reference": "Patient/ebb1f0e2-4fa6-4889-a43b-9cda3c737078"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "2013-04-21T22:26:37+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "95d8ae15-aa5c-467c-84ec-b4d51d4b3e28",
      "subject": {
        "reference": "Patient/71304c2e-64b0-4b57-91f6-2c1ff771d72c"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "1985-01-17T18:42:18+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "abc9ea4f-6eb2-44b6-8f02-e16cb056b5ca",
      "subject": {
        "reference": "Patient/881f534f-d041-425d-a542-cbf669f43e18"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "2003-01-15T13:06:23+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "42e5f6d1-4f64-43d8-96d3-a1c213aca769",
      "subject": {
        "reference": "Patient/9eb43ac3-7c1e-4e25-94cd-4b2c43f7234e"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "2009-06-27T07:38:43+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "18c1215b-8fc4-466e-a15c-417004fb2535",
      "subject": {
        "reference": "Patient/9bf59715-246f-4ce2-85a6-58c1757f8d21"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "1982-02-01T16:34:40+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "69d55451-9037-480d-afdb-19f84d959b53",
      "subject": {
        "reference": "Patient/0e61c3ad-d11e-4080-a6aa-cac89cae4e37"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "1991-04-18T05:09:36+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "792c8847-975e-435d-bdff-0a22a0228fe8",
      "subject": {
        "reference": "Patient/948e608d-e408-4788-8cf2-8157c49c2940"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "2019-03-05T20:32:26+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "c54489bc-e98b-4797-87e4-6f9592a0543e",
      "subject": {
        "reference": "Patient/9efca32a-04f9-438a-990c-a287e8b62aac"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "1951-04-23T23:11:46+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "37b8d49f-af81-4f2a-ba2c-cd16029b942d",
      "subject": {
        "reference": "Patient/d483a480-4a42-428c-bf9b-fdbaddf78e21"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "2000-01-17T07:36:34+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "c0a1d2ef-797c-4b23-ba45-90fad1d17344",
      "subject": {
        "reference": "Patient/5fd069dc-e337-4062-9722-a732c655c17d"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "1963-07-27T19:20:06+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "9440dbf4-f93c-47e9-96c6-9cad9a6d9506",
      "subject": {
        "reference": "Patient/bd6f9f37-7295-4cd6-b212-134afcef1253"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "1998-03-21T21:37:43+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "9703dd83-e507-4ebe-96e6-b785661c9f60",
      "subject": {
        "reference": "Patient/7f445dad-0907-4a3e-81a5-ad75ecdff752"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "2005-12-13T11:02:24+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "75448cb2-1aca-4b6d-bb56-2217b19d9c7c",
      "subject": {
        "reference": "Patient/132f48ba-1d72-4e0c-a220-91786d7501aa"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "2017-11-10T00:14:49+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "b0c5762a-4a6a-4799-98cf-89156b62602b",
      "subject": {
        "reference": "Patient/f12a1a30-9ce0-48dd-91de-ce5c3fcf8828"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "1978-04-15T07:44:08+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "3e2b6e19-1397-4f56-abf5-adbccf36b967",
      "subject": {
        "reference": "Patient/3e26303e-a72a-4c60-866f-e5b81a9f1989"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "1963-07-22T22:39:41+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "5f5383f0-ec9c-4c29-b49a-c080648cb13c",
      "subject": {
        "reference": "Patient/af0e5952-c2ac-44fc-a896-36e9e30c2097"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "2007-04-19T19:56:12+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "bb93ac67-f7ba-4d92-a744-c6d016f776e9",
      "subject": {
        "reference": "Patient/ffd502c9-23e1-4f8f-bc8a-87373acad280"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "2001-11-19T17:17:39+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "b846e0f3-b308-4497-905f-af4c600aa434",
      "subject": {
        "reference": "Patient/fa064acf-b7f1-4279-83d3-7a94686da7ba"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "1962-05-05T02:39:21+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "eeec214d-af48-4ba3-8a92-00b10ef96102",
      "subject": {
        "reference": "Patient/efe58fa2-c6df-4a71-8e37-06c5e4f1e2b8"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "2007-08-03T00:26:57+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "ac0f8b2e-6be5-4473-9810-a0c2168204ea",
      "subject": {
        "reference": "Patient/1ac8947d-038f-4cc7-81fa-a32e694187e8"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "1994-05-16T16:42:48+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "f8fa5f28-e6a4-428c-95ff-c28229e6bf58",
      "subject": {
        "reference": "Patient/7099b4c5-6f47-4293-9690-f2afb23b9dd6"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "1981-08-30T00:41:27+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "0b8d8e19-32e9-469c-abcc-d7fd212a8700",
      "subject": {
        "reference": "Patient/4d58465c-4703-4c36-ab8f-2f935dc4bee7"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "1995-10-26T17:37:12+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "d0561e5b-e81e-4405-b9ce-7447baac01db",
      "subject": {
        "reference": "Patient/5db1e775-33a3-47fb-8954-d97cd43e8074"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "2007-07-22T18:29:30+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "c21b61f1-43d5-49b6-88d5-b8abea278a0c",
      "subject": {
        "reference": "Patient/148f10e1-2182-4297-ac1f-39bb9eca6f1a"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "2019-06-25T04:22:04+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "1f523b43-0166-45cf-b752-523524112b3d",
      "subject": {
        "reference": "Patient/1d4685c4-e83e-4a73-a8e5-147a4c0b4ed4"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "1957-06-24T23:11:46+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "91a2d625-f886-4902-9507-215e7d08d5e4",
      "subject": {
        "reference": "Patient/75a4631d-2cd8-4c77-afb6-740bad394ea1"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "1988-08-22T07:57:54+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "0b1f17ec-e88b-4d74-9d3c-abd13001d1e4",
      "subject": {
        "reference": "Patient/ec6a274e-5090-45c9-b153-e40418f6fa3d"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "1987-07-23T19:27:45+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "8a69a588-1cf3-49b3-bcc1-29201eca2383",
      "subject": {
        "reference": "Patient/bae2bf00-eca1-47fb-bb43-272f8de1449c"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "2020-01-31T19:28:31+00:00"
    },
    {
      "resourceType": "Condition",
      "id": "5cfeb498-f18f-42ba-ab9e-c34b44d5f450",
      "subject": {
        "reference": "Patient/12edb5a0-ec71-45b7-aff6-0fef1c382881"
      },
      "code": {
        "coding": [
          {
            "system": "http://snomed.info/sct",
            "code": "44054006",
            "display": "Diabetes"
          }
        ],
        "text": "Diabetes"
      },
      "onsetDateTime": "1973-11-15T09:15:59+00:00"
    }
  ],
  "observations": {
    "948e608d-e408-4788-8cf2-8157c49c2940": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-948e608d",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/948e608d-e408-4788-8cf2-8157c49c2940"
        },
        "effectiveDateTime": "2021-03-02T20:32:26+00:00",
        "valueQuantity": {
          "value": 3.0102167723507556,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "1ac8947d-038f-4cc7-81fa-a32e694187e8": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-1ac8947d",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/1ac8947d-038f-4cc7-81fa-a32e694187e8"
        },
        "effectiveDateTime": "2013-11-18T16:42:48+00:00",
        "valueQuantity": {
          "value": 5.296006368883365,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "881f534f-d041-425d-a542-cbf669f43e18": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-881f534f",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/881f534f-d041-425d-a542-cbf669f43e18"
        },
        "effectiveDateTime": "2020-10-21T13:06:23+00:00",
        "valueQuantity": {
          "value": 5.4413479558389986,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "bd6f9f37-7295-4cd6-b212-134afcef1253": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-bd6f9f37",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/bd6f9f37-7295-4cd6-b212-134afcef1253"
        },
        "effectiveDateTime": "2021-03-13T21:37:43+00:00",
        "valueQuantity": {
          "value": 2.8697750319521544,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "fa064acf-b7f1-4279-83d3-7a94686da7ba": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-fa064acf",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/fa064acf-b7f1-4279-83d3-7a94686da7ba"
        },
        "effectiveDateTime": "2021-03-27T02:39:21+00:00",
        "valueQuantity": {
          "value": 5.420683972200942,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "7099b4c5-6f47-4293-9690-f2afb23b9dd6": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-7099b4c5",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/7099b4c5-6f47-4293-9690-f2afb23b9dd6"
        },
        "effectiveDateTime": "2021-01-31T00:41:27+00:00",
        "valueQuantity": {
          "value": 2.4307078680568486,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "af0e5952-c2ac-44fc-a896-36e9e30c2097": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-af0e5952",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/af0e5952-c2ac-44fc-a896-36e9e30c2097"
        },
        "effectiveDateTime": "2020-07-02T19:56:12+00:00",
        "valueQuantity": {
          "value": 6.093822434075272,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "efe58fa2-c6df-4a71-8e37-06c5e4f1e2b8": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-efe58fa2",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/efe58fa2-c6df-4a71-8e37-06c5e4f1e2b8"
        },
        "effectiveDateTime": "2021-01-22T00:26:57+00:00",
        "valueQuantity": {
          "value": 7.262022744084565,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "ffd502c9-23e1-4f8f-bc8a-87373acad280": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-ffd502c9",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/ffd502c9-23e1-4f8f-bc8a-87373acad280"
        },
        "effectiveDateTime": "2020-09-07T17:17:39+00:00",
        "valueQuantity": {
          "value": 6.109457325824228,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "9eb43ac3-7c1e-4e25-94cd-4b2c43f7234e": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-9eb43ac3",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/9eb43ac3-7c1e-4e25-94cd-4b2c43f7234e"
        },
        "effectiveDateTime": "2020-06-20T07:38:43+00:00",
        "valueQuantity": {
          "value": 6.118428408005564,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "1d4685c4-e83e-4a73-a8e5-147a4c0b4ed4": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-1d4685c4",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/1d4685c4-e83e-4a73-a8e5-147a4c0b4ed4"
        },
        "effectiveDateTime": "1976-03-22T23:11:46+00:00",
        "valueQuantity": {
          "value": 3.837063414120573,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "5db1e775-33a3-47fb-8954-d97cd43e8074": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-5db1e775",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/5db1e775-33a3-47fb-8954-d97cd43e8074"
        },
        "effectiveDateTime": "2020-09-20T18:29:30+00:00",
        "valueQuantity": {
          "value": 6.091997925244769,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "9bf59715-246f-4ce2-85a6-58c1757f8d21": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-9bf59715",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/9bf59715-246f-4ce2-85a6-58c1757f8d21"
        },
        "effectiveDateTime": "2021-01-11T16:34:40+00:00",
        "valueQuantity": {
          "value": 5.314300264456399,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "f12a1a30-9ce0-48dd-91de-ce5c3fcf8828": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-f12a1a30",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/f12a1a30-9ce0-48dd-91de-ce5c3fcf8828"
        },
        "effectiveDateTime": "2009-02-07T07:44:08+00:00",
        "valueQuantity": {
          "value": 3.5999999999999996,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "0e61c3ad-d11e-4080-a6aa-cac89cae4e37": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-0e61c3ad",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/0e61c3ad-d11e-4080-a6aa-cac89cae4e37"
        },
        "effectiveDateTime": "2020-09-17T05:09:36+00:00",
        "valueQuantity": {
          "value": 3.0675834361733108,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "d483a480-4a42-428c-bf9b-fdbaddf78e21": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-d483a480",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/d483a480-4a42-428c-bf9b-fdbaddf78e21"
        },
        "effectiveDateTime": "2021-03-15T07:36:34+00:00",
        "valueQuantity": {
          "value": 7.0196643089439625,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "12edb5a0-ec71-45b7-aff6-0fef1c382881": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-12edb5a0",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/12edb5a0-ec71-45b7-aff6-0fef1c382881"
        },
        "effectiveDateTime": "2021-03-11T09:15:59+00:00",
        "valueQuantity": {
          "value": 4.031917522507655,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "ebb1f0e2-4fa6-4889-a43b-9cda3c737078": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-ebb1f0e2",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/ebb1f0e2-4fa6-4889-a43b-9cda3c737078"
        },
        "effectiveDateTime": "2019-04-28T22:26:37+00:00",
        "valueQuantity": {
          "value": 6.454976133754152,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "ec6a274e-5090-45c9-b153-e40418f6fa3d": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-ec6a274e",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/ec6a274e-5090-45c9-b153-e40418f6fa3d"
        },
        "effectiveDateTime": "2020-09-03T19:27:45+00:00",
        "valueQuantity": {
          "value": 5.998967835047721,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "4d58465c-4703-4c36-ab8f-2f935dc4bee7": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-4d58465c",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/4d58465c-4703-4c36-ab8f-2f935dc4bee7"
        },
        "effectiveDateTime": "2021-02-18T17:37:12+00:00",
        "valueQuantity": {
          "value": 3.011528783950294,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "5fd069dc-e337-4062-9722-a732c655c17d": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-5fd069dc",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/5fd069dc-e337-4062-9722-a732c655c17d"
        },
        "effectiveDateTime": "1981-01-03T19:20:06+00:00",
        "valueQuantity": {
          "value": 2.3887090909769846,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "bae2bf00-eca1-47fb-bb43-272f8de1449c": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-bae2bf00",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/bae2bf00-eca1-47fb-bb43-272f8de1449c"
        },
        "effectiveDateTime": "2020-10-09T19:28:31+00:00",
        "valueQuantity": {
          "value": 7.4987279015619155,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "9efca32a-04f9-438a-990c-a287e8b62aac": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-9efca32a",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/9efca32a-04f9-438a-990c-a287e8b62aac"
        },
        "effectiveDateTime": "1979-09-24T23:11:46+00:00",
        "valueQuantity": {
          "value": 3.014329604451408,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "71304c2e-64b0-4b57-91f6-2c1ff771d72c": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-71304c2e",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/71304c2e-64b0-4b57-91f6-2c1ff771d72c"
        },
        "effectiveDateTime": "2020-11-05T18:42:18+00:00",
        "valueQuantity": {
          "value": 5.905981569202394,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "ecaea95c-46de-4ac9-a58d-4847d1a3e574": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-ecaea95c",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/ecaea95c-46de-4ac9-a58d-4847d1a3e574"
        },
        "effectiveDateTime": "1969-02-22T19:20:06+00:00",
        "valueQuantity": {
          "value": 3.8967682333517244,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "75a4631d-2cd8-4c77-afb6-740bad394ea1": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-75a4631d",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/75a4631d-2cd8-4c77-afb6-740bad394ea1"
        },
        "effectiveDateTime": "2008-02-25T07:57:54+00:00",
        "valueQuantity": {
          "value": 5.436466680400847,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "7f445dad-0907-4a3e-81a5-ad75ecdff752": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-7f445dad",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/7f445dad-0907-4a3e-81a5-ad75ecdff752"
        },
        "effectiveDateTime": "2021-01-12T11:02:24+00:00",
        "valueQuantity": {
          "value": 7.056126343122046,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "132f48ba-1d72-4e0c-a220-91786d7501aa": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-132f48ba",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/132f48ba-1d72-4e0c-a220-91786d7501aa"
        },
        "effectiveDateTime": "2021-03-12T00:14:49+00:00",
        "valueQuantity": {
          "value": 3.028126053975507,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "3e26303e-a72a-4c60-866f-e5b81a9f1989": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-3e26303e",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/3e26303e-a72a-4c60-866f-e5b81a9f1989"
        },
        "effectiveDateTime": "1984-10-08T22:39:41+00:00",
        "valueQuantity": {
          "value": 2.5903553371032935,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ],
    "148f10e1-2182-4297-ac1f-39bb9eca6f1a": [
      {
        "resourceType": "Observation",
        "id": "obs-hba1c-148f10e1",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "4548-4",
              "display": "Hemoglobin A1c/Hemoglobin.total in Blood"
            }
          ],
          "text": "Hemoglobin A1c/Hemoglobin.total in Blood"
        },
        "subject": {
          "reference": "Patient/148f10e1-2182-4297-ac1f-39bb9eca6f1a"
        },
        "effectiveDateTime": "2019-06-25T04:22:04+00:00",
        "valueQuantity": {
          "value": 7.564455213778822,
          "unit": "%",
          "system": "http://unitsofmeasure.org",
          "code": "%"
        }
      }
    ]
  }
}

# ---- Mini FHIR Server ----
app = Flask(__name__)

@app.route("/metadata")
def metadata():
    return jsonify({
        "resourceType": "CapabilityStatement",
        "status": "active",
        "fhirVersion": "4.0.0",
        "format": ["json"],
        "rest": [{"mode": "server", "resource": [
            {"type": "Patient"}, {"type": "Condition"}, {"type": "Observation"}
        ]}]
    })

@app.route("/Patient/<patient_id>")
def get_patient(patient_id):
    patient = FHIR_DATA["patients"].get(patient_id)
    if patient:
        return jsonify(patient)
    return jsonify({"resourceType": "OperationOutcome", "issue": [{"severity": "error", "code": "not-found"}]}), 404

@app.route("/Patient")
def search_patients():
    count = int(flask_request.args.get("_count", 50))
    all_patients = list(FHIR_DATA["patients"].values())
    results = all_patients[:count]
    return jsonify({
        "resourceType": "Bundle",
        "type": "searchset",
        "total": len(all_patients),
        "entry": [{"resource": p} for p in results]
    })

@app.route("/Condition")
def search_conditions():
    code = flask_request.args.get("code", "")
    subject = flask_request.args.get("subject", "")
    count = int(flask_request.args.get("_count", 50))
    results = FHIR_DATA["conditions"]
    # Filter by code (support both bare code and system|code)
    if code:
        code_value = code.split("|")[-1] if "|" in code else code
        results = [c for c in results
            if any(coding["code"] == code_value for coding in c["code"]["coding"])]
    # Filter by subject
    if subject:
        results = [c for c in results if c["subject"]["reference"] == subject]
    results = results[:count]
    return jsonify({
        "resourceType": "Bundle",
        "type": "searchset",
        "total": len(results),
        "entry": [{"resource": c} for c in results]
    })

@app.route("/Observation")
def search_observations():
    subject = flask_request.args.get("subject", "")
    code = flask_request.args.get("code", "")
    count = int(flask_request.args.get("_count", 50))
    # Extract patient ID from subject param
    patient_id = subject.replace("Patient/", "") if subject else ""
    obs_list = FHIR_DATA["observations"].get(patient_id, [])
    # Filter by code if provided
    if code:
        code_value = code.split("|")[-1] if "|" in code else code
        obs_list = [o for o in obs_list
            if any(coding["code"] == code_value for coding in o["code"]["coding"])]
    # Sort by date descending (most recent first)
    obs_list = sorted(obs_list, key=lambda o: o.get("effectiveDateTime", ""), reverse=True)
    obs_list = obs_list[:count]
    return jsonify({
        "resourceType": "Bundle",
        "type": "searchset",
        "total": len(obs_list),
        "entry": [{"resource": o} for o in obs_list]
    })

# Start server in background thread
import logging
logging.getLogger('werkzeug').setLevel(logging.ERROR)
threading.Thread(target=lambda: app.run(port=5050, use_reloader=False), daemon=True).start()
time.sleep(1)  # Wait for server to start

# ---- Configuration ----
FHIR_BASE = "http://localhost:5050"

def show_json(data, max_lines=30):
    """Pretty-print JSON, truncated for readability."""
    text = json.dumps(data, indent=2)
    lines = text.split('\n')
    if len(lines) > max_lines:
        print('\n'.join(lines[:max_lines]))
        print(f'\n... ({len(lines) - max_lines} more lines)')
    else:
        print(text)

# Verify the server is running
resp = requests.get(f"{FHIR_BASE}/metadata", params={"_format": "json"}, timeout=10)
if resp.status_code == 200:
    fhir_version = resp.json().get("fhirVersion", "unknown")
    print(f"✅ Local FHIR server running with cached Synthea data")
    print(f"   Server: {FHIR_BASE}")
    print(f"   FHIR version: {fhir_version}")
    print(f"   Patients loaded: 30 (Type 2 Diabetes cohort)")
else:
    print(f"❌ Server not responding. Status: {resp.status_code}")

## Step 0: Your First FHIR Query

Let's start by fetching a few Patient resources to see what FHIR data looks like.

In [ ]:
# ============================================================
# YOUR FIRST FHIR QUERY — Fetching Patient resources
# ============================================================
resp = requests.get(f"{FHIR_BASE}/Patient",
    params={"_count": 3, "_format": "json"})
bundle = resp.json()

print(f"Response type: {bundle['resourceType']}")  # Always 'Bundle' for search results
print(f"Total patients on server: {bundle.get('total', 'unknown')}")
print(f"Entries returned: {len(bundle.get('entry', []))}")
print()
print("--- First Patient Resource ---")
first_patient = bundle["entry"][0]["resource"]
show_json(first_patient)

### 🔍 What Did We Just See?

The FHIR server returned a **Bundle** — a container for search results.

- `resourceType: "Bundle"` — this is a search result container
- `total` — how many resources matched on the entire server
- `entry` — an array of results, each containing a `resource`

Inside each **Patient** resource:
- `id` — the unique identifier. Other resources use this to REFERENCE this patient.
- `name` — array with `given` (first name) and `family` (last name)
- `birthDate`, `gender` — demographics

**The URL pattern:** `{server}/Patient?_count=3` means "give me up to 3 Patient resources."
You'll use this same pattern with Condition and Observation next.

## ✏️ Step 1: Find Patients with Type 2 Diabetes

Now you'll use Claude to write a FHIR query. Open the **Claude web interface** and enter this prompt:

> Write Python code using the `requests` library to search for Condition
> resources with SNOMED CT code 44054006 (Type 2 diabetes) on a FHIR server.
> The server base URL is already stored in a variable called `FHIR_BASE`.
> Limit to 30 results. For each condition found, extract and print:
> - The condition resource ID
> - The patient reference (from subject.reference)
> - The display name of the condition
> - The onset date (from onsetDateTime)
>
> Also collect all unique patient references into a Python set called `patient_refs`.

**Paste Claude's response in the cell below and run it (Shift+Enter).**

In [ ]:
# Paste Claude's code here and run it


In [ ]:
# ============================================================
# VERIFICATION — Check your Condition search results
# ============================================================
try:
    print(f"✅ Found {len(patient_refs)} unique patients with Type 2 diabetes")
    print(f"   Example references: {list(patient_refs)[:5]}")
    print()
    # Extract just the IDs for the next step
    patient_ids = [ref.split('/')[-1] for ref in patient_refs if '/' in ref]
    print(f"   Extracted patient IDs: {patient_ids[:5]}")
    print(f"\n   Next: we'll FOLLOW these references to get patient demographics.")
except NameError:
    print("⚠️  Variable 'patient_refs' not found.")
    print("   Make sure your code creates a set called 'patient_refs'")
    print("   containing strings like 'Patient/abc123'")
    print()
    print("   If Claude used a different variable name, rename it and re-run,")
    print("   or uncomment the fallback below:")
    print()
    print("   # --- FALLBACK ---")
    print("   # resp = requests.get(f'{FHIR_BASE}/Condition',")
    print("   #     params={'code': '44054006', '_count': 30, '_format': 'json'})")
    print("   # bundle = resp.json()")
    print("   # patient_refs = set()")
    print("   # for entry in bundle.get('entry', []):")
    print("   #     ref = entry['resource'].get('subject', {}).get('reference', '')")
    print("   #     if ref: patient_refs.add(ref)")
    print("   # patient_ids = [ref.split('/')[-1] for ref in patient_refs]")

## ✏️ Step 2: Get Patient Demographics

Each Condition resource points to a Patient via `subject.reference` (e.g., `Patient/abc123`).
Now we follow those references to get each patient's name, birthdate, and gender.

Ask Claude:

> I have a Python list called `patient_ids` containing FHIR patient IDs like
> `["abc123", "def456"]`. Write Python code that fetches each Patient resource
> from a FHIR server. The server base URL is in a variable called `FHIR_BASE`,
> so fetch from `f"{FHIR_BASE}/Patient/{id}"`. Extract their full name
> (combining given and family name), birth date, and gender. Store the results
> in a list of dictionaries called `patients` where each dict has keys:
> `"id"`, `"name"`, `"birthDate"`, `"gender"`. Print each patient as you fetch them.

**Paste Claude's code below.**

In [ ]:
# Paste Claude's code here and run it


In [ ]:
# ============================================================
# PATIENT DEMOGRAPHICS TABLE
# ============================================================
try:
    df_patients = pd.DataFrame(patients)
    print(f"✅ Retrieved demographics for {len(df_patients)} patients:\n")
    display(df_patients)
    print(f"\nNotice: each patient has an 'id' — we'll use this to search for their lab results.")
except NameError:
    print("⚠️  Variable 'patients' not found.")
    print("   Make sure your code creates a list called 'patients'.")
    print("   Each item: {'id': '...', 'name': '...', 'birthDate': '...', 'gender': '...'}")

## ✏️ Step 3: Retrieve HbA1c Lab Values

This is the critical step. For EACH patient, we search for Observation resources
with LOINC code **4548-4** (HbA1c). We want only the most recent result.

Ask Claude:

> I have a Python list called `patients` where each item is a dictionary with
> an `"id"` key containing a FHIR patient ID. For each patient, write Python
> code to search for Observation resources on a FHIR server. The server base
> URL is in a variable called `FHIR_BASE`. Search at
> `f"{FHIR_BASE}/Observation"` with these URL parameters:
> - `subject`: `Patient/{id}`
> - `code`: `4548-4`
> - `_sort`: `-date`
> - `_count`: `1`
>
> Extract the date (`effectiveDateTime`), numeric value (`valueQuantity.value`),
> and unit (`valueQuantity.unit`) from the most recent observation.
> Store results in a list called `observations` where each dict has keys:
> `"patient_id"`, `"date"`, `"value"`, `"unit"`. If a patient has no HbA1c
> observation, include them with value `"N/A"`. Print progress.

**Paste Claude's code below.**

In [ ]:
# Paste Claude's code here and run it


In [ ]:
# ============================================================
# COMBINED ANALYSIS — Identify Poor Glycemic Control
# ============================================================
try:
    df_obs = pd.DataFrame(observations)
    df_patients = pd.DataFrame(patients)
    df_patients_copy = df_patients.copy()
    df_patients_copy['id'] = df_patients_copy['id'].astype(str)
    df_obs['patient_id'] = df_obs['patient_id'].astype(str)

    df_merged = df_obs.merge(df_patients_copy, left_on='patient_id', right_on='id', how='left')

    # Convert to numeric
    df_merged['hba1c_numeric'] = pd.to_numeric(df_merged['value'], errors='coerce')

    # Flag poor control
    def control_flag(x):
        if pd.isna(x): return '⚪ No data'
        if x > 7.0: return '🔴 Poor control'
        if x >= 6.5: return '🟡 Diabetic range'
        return '🟢 Below threshold'

    df_merged['glycemic_control'] = df_merged['hba1c_numeric'].apply(control_flag)

    display_cols = [c for c in ['name','birthDate','gender','date','value','unit','glycemic_control']
                    if c in df_merged.columns]
    print('📊 Patients with Type 2 Diabetes — HbA1c Results:\n')
    display(df_merged[display_cols])

    has_data = df_merged['hba1c_numeric'].notna()
    poor = (df_merged['hba1c_numeric'] > 7.0) & has_data
    print(f'\n📈 Summary:')
    print(f'   Total diabetic patients: {len(df_merged)}')
    print(f'   With HbA1c data: {has_data.sum()}')
    print(f'   🔴 Poor control (>7.0%): {poor.sum()}')
    print(f'   🟢 Adequate: {(has_data & ~poor).sum()}')
    print(f'   ⚪ No data: {(~has_data).sum()}')

    if poor.sum() > 0:
        print(f'\n   Patients needing follow-up:')
        for _, row in df_merged[poor].iterrows():
            print(f"     • {row.get('name','?')}: HbA1c = {row['value']}%")

except NameError as e:
    print(f'⚠️  Error: {e}')
    print('   Make sure you ran Steps 1-3 successfully.')
except Exception as e:
    print(f'⚠️  Unexpected error: {e}')
    print('   Ask Claude to help debug.')

## ✏️ Step 4: Generate a Clinical Summary

We now have structured data in a table. The final step: translate this into
a narrative a clinician or patient could read.

**Copy the table output above** and paste it into the Claude web interface with:

> Here is a table of patients with Type 2 diabetes and their most recent HbA1c
> values. Write a brief clinical summary suitable for a care coordinator.
> Identify patients with poor glycemic control (HbA1c > 7.0%) and note they may
> need follow-up. Format as 2-3 short paragraphs. Use ONLY the data in the
> table — do not add any information that is not present.

**Paste Claude's summary in the markdown cell below.**

### Clinical Summary

*Paste Claude's summary here...*

## 🧠 Session 1 Reflection

You just manually executed a **three-step clinical data pipeline**:

1. **Condition search** → Found patients with Type 2 diabetes (SNOMED CT: 44054006)
2. **Patient lookup** → Retrieved demographics by following `subject.reference`
3. **Observation search** → Got HbA1c lab values (LOINC: 4548-4) for each patient

You used an LLM (Claude) in two distinct roles:
- **Code generation** — Claude wrote the Python/FHIR queries
- **Summarization** — Claude translated structured data into clinical narrative

**Key insight:** The LLM never touched the FHIR server directly. YOUR CODE did
the querying. The LLM helped you *write* the code and *interpret* the results.
This separation between planning/interpretation (LLM) and execution (code) is
critical for safety and correctness in clinical systems.

**Note:** This backup notebook used a local FHIR server with cached Synthea data.
The code you wrote works identically with the real SMART FHIR server — only the
`FHIR_BASE` URL differs.

**Next session:** What if the LLM could orchestrate these same steps
*autonomously* — deciding which queries to run and in what order? That's called
**tool use**, and it's the foundation of AI agents.

### 💾 Save this notebook — you'll reference it in Session 2.